# Model Development — Churn Prediction

Build and compare two churn models, and select one to carry forward.

The previous notebook, `02_schema_build.ipynb`, built the database and derived the features. This notebook uses them. Two models are built: a logistic regression and a gradient boosting classifier. Both are evaluated under identical conditions, and one is selected.

**The selection is decided in advance, and on stated grounds.** Logistic regression is the intended production model because its decisions can be explained. Each feature carries a single number showing the direction and size of its effect, which means any individual prediction can be traced back to the reasons behind it. This is a requirement in regulated financial environments, where a customer or a regulator may ask why a particular decision was made.

Gradient boosting is built regardless, because a stated preference for the simpler model only carries weight if the more complex alternative was measured rather than avoided.

---

## 1. Load the feature set

Read the completed feature and value table from the database into Python.

No data preparation is performed here. Every feature was derived in SQL and arrives ready for use. This is the intended division of work: the database performs the transformation, and the modelling layer consumes the result.

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect("../telco.duckdb")

df = con.execute("SELECT * FROM vw_customer_value_segments").df()

print(df.shape)
df.head()

(7043, 18)


,customer_id,tenure_months,monthly_charges,total_charges,contract,internet_service,churn_value,tenure_band,has_internet,has_phone,addon_count,contract_risk,automatic_payment,spend_band,expected_remaining_months,historic_margin,clv,value_decile
0,7569-NMZYQ,72,118.75,8672.45,Two year,Fiber optic,0,Loyal,1,1,6,1,1,High,36,2601.74,1282.50,1
1,8984-HPEMB,71,118.65,8477.60,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2543.28,1281.42,1
2,5989-AXPUC,68,118.60,7990.05,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2397.02,1280.88,1
3,9924-JPRMC,72,118.20,8547.15,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2564.15,1276.56,1
4,3810-DVDQQ,72,117.60,8308.90,Two year,Fiber optic,0,Loyal,1,1,6,1,1,High,36,2492.67,1270.08,1


**Result:** 7,043 rows and 18 columns, matching the record count established during profiling.

## 2. Select the features the model is permitted to use

Not every available column belongs in the model. Four groups are excluded, for four different reasons.

**The identifier.** `customer_id` labels a customer but describes nothing about them.

**The target.** `churn_value` is the outcome being predicted and cannot also be an input.

**The customer value fields** — `clv`, `historic_margin`, `expected_remaining_months` and `value_decile`. These are excluded deliberately and the reason is central to the project. Churn risk and customer value are two separate questions. The model answers the first; the economics layer answers the second; the two are combined afterwards to decide where to act. If value were included as a model input, that separation would no longer exist and the argument could not be made.

**Duplicated fields.** `tenure_band` repeats `tenure_months` in grouped form, `spend_band` repeats `monthly_charges`, `contract` repeats `contract_risk` in words, and `has_internet` is already contained within `internet_service`. Supplying the same information twice makes the model's coefficients unstable and harder to read, which would undermine the reason for choosing logistic regression. The banded fields remain available for reporting, where grouping is useful.

`internet_service` holds text values, which a model cannot read. `pd.get_dummies` converts them into numeric columns. `drop_first=True` removes one of the three resulting columns, because with two of them known the third is implied — including all three would reintroduce the duplication just described.

In [3]:
target = "churn_value"

features = [
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "has_phone",
    "addon_count",
    "contract_risk",
    "automatic_payment",
    "internet_service",
]

X = pd.get_dummies(df[features], columns=["internet_service"], drop_first=True)
y = df[target]

print(X.shape)
X.head()

(7043, 9)


,tenure_months,monthly_charges,total_charges,has_phone,addon_count,contract_risk,automatic_payment,internet_service_Fiber optic,internet_service_No
0,72,118.75,8672.45,1,6,1,1,True,False
1,71,118.65,8477.60,1,6,1,0,True,False
2,68,118.60,7990.05,1,6,1,0,True,False
3,72,118.20,8547.15,1,6,1,0,True,False
4,72,117.60,8308.90,1,6,1,1,True,False


**Result:** nine features. DSL is the omitted internet category, so the two remaining internet columns are read as differences from DSL.

## 3. Separate the data into training and test sets

Divide the customers into two groups. The model learns from one group and is measured on the other, which it never sees during training.

This separation is what makes the measurement meaningful. A model evaluated on the same records it learned from can simply reproduce what it memorised, and would report a level of performance it could not repeat on new customers.

Three settings are applied:

- **`test_size=0.25`** holds back a quarter of customers for measurement.
- **`stratify=y`** ensures both groups contain the same proportion of churned customers as the full dataset, 26.5%. Without this, a random division could produce a test set with an unrepresentative share of churners, and the results would not be reliable.
- **`random_state=42`** fixes the division so that it is identical every time the notebook runs. Anyone re-running this work obtains the same numbers and can verify them. The specific value is arbitrary and conventional.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(5282, 9) (1761, 9)
0.2654297614539947 0.26519023282226006


**Result:** 5,282 customers for training and 1,761 for testing. Both groups show a churn rate of 0.265, confirming the proportions were preserved.

## 4. Train the logistic regression

Three components are used, each for a specific reason.

**`StandardScaler`** places all features on a common scale. The features are measured in very different units — total charges run into the thousands, while the add-on count runs from zero to six. Without scaling, the model would treat the larger numbers as more influential purely because of their size. Scaling ensures the coefficients reflect genuine influence rather than the unit of measurement.

**`Pipeline`** combines the scaling and the model into a single object. This is a correctness measure rather than a convenience. The scaler learns its adjustments from the training data only and then applies them to the test data. Performing the two steps separately makes it easy to scale all data together, which would allow information from the test set to influence training.

**`class_weight="balanced"`** addresses the imbalance in the data. Only 26.5% of customers churned, so a model optimising for overall accuracy could score 73.5% by predicting that nobody ever churns. This setting instructs the model to treat a missed churner as a proportionately more serious error.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

logreg.fit(X_train, y_train)
print("trained")

trained


## 5. Measure performance on the held-back customers

Four measures are used.

**The confusion matrix** is a table of four counts: customers correctly predicted to stay, customers wrongly flagged as leaving, churners the model missed, and churners it correctly identified.

**Precision** is the share of flagged customers who genuinely churned. Low precision means retention budget is spent on customers who were never going to leave.

**Recall** is the share of actual churners the model identified. Low recall means customers are lost without warning.

Precision and recall move against one another. The balance between them is a commercial decision rather than a technical one, and it is the decision the customer value layer is built to inform.

**ROC AUC** summarises, in a single figure between 0.5 and 1.0, how well the model separates churners from non-churners across all possible thresholds. Approximately 0.84 is the established range for this dataset. A figure close to 1.0 would indicate that information about the outcome had leaked into the inputs.

`predict_proba` returns a probability for each customer rather than a yes or no answer. The probability is retained, because the economics layer requires a graded measure of risk rather than a binary label.

In [6]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred  = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba), 3))

[[932 362]
 [ 97 370]]

              precision    recall  f1-score   support

           0      0.906     0.720     0.802      1294
           1      0.505     0.792     0.617       467

    accuracy                          0.739      1761
   macro avg      0.706     0.756     0.710      1761
weighted avg      0.800     0.739     0.753      1761

ROC AUC: 0.84


**Result:** ROC AUC of 0.840, within the expected range.

Of the 1,761 held-back customers, 467 genuinely churned. The model identified 370 of them and missed 97, giving recall of 79.2%. It flagged 732 customers in total, meaning 362 flags were raised on customers who did not leave, giving precision of 50.5%.

Stated commercially: approximately four in five departing customers are identified, and for each one identified, roughly one additional customer is flagged unnecessarily. Whether that represents a sound investment depends on the cost of the intervention relative to the value of the customer — which is precisely the question the lifetime value layer answers.

**Accuracy is 73.9%**, which is marginally below the 73.5% obtainable by predicting that no customer ever churns. This is retained in the notebook rather than omitted, because it demonstrates directly why accuracy is an inappropriate measure for imbalanced problems of this kind.

> **Insight — half of any retention spend will land on customers who were never leaving.** Precision of 50.5% is not a fault to be corrected; it is the working condition. It means the decision of whether to intervene cannot be made on the model's output alone, and must be weighed against what each customer is worth.

## 6. Examine the coefficients

Print the coefficient attached to each feature. This is the practical benefit of logistic regression: the model's reasoning is fully visible.

A positive coefficient indicates the feature increases the likelihood of churn; a negative coefficient indicates it reduces the likelihood. Because all features were placed on a common scale, the magnitudes can be compared directly with one another.

In [7]:
coefs = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": logreg.named_steps["model"].coef_[0]
}).sort_values("coefficient", ascending=False)

coefs

,feature,coefficient
1,monthly_charges,1.835196
5,contract_risk,0.733298
2,total_charges,0.573623
8,internet_service_No,0.263224
6,automatic_payment,-0.172772
7,internet_service_Fiber optic,-0.269150
3,has_phone,-0.594036
4,addon_count,-0.821713
0,tenure_months,-1.218183


**Result:** most coefficients are consistent with the findings established in SQL, but two are not.

**Consistent with expectations:**

| Feature | Coefficient | Interpretation |
|---|---|---|
| `monthly_charges` | +1.84 | Higher monthly cost increases churn |
| `contract_risk` | +0.73 | Month-to-month customers churn more |
| `tenure_months` | −1.22 | Longer-standing customers churn less |
| `addon_count` | −0.82 | Each additional service held reduces churn |

> **Insight — price is the strongest single driver of churn.** `monthly_charges` carries the largest coefficient of any feature, ahead of contract type and tenure. Retention conversations that do not address the bill are addressing the second-order problem.

**Inconsistent, and requiring investigation:**

`total_charges` at +0.57 indicates that customers who have paid more over their lifetime are more likely to leave. This directly contradicts `tenure_months`, which indicates the opposite. Both cannot be correct.

`internet_service_Fiber optic` at −0.27 indicates that fibre customers churn less, which is contrary to the known behaviour of this dataset.

**The cause is shared information between features.** `total_charges` is approximately `monthly_charges` multiplied by `tenure_months` — it introduces no new information, being a product of two features already present. When features carry overlapping information, the model distributes the effect between them in an arbitrary manner. The predictions remain sound, but the individual coefficients cease to be reliable — and the coefficients are the entire justification for selecting this model.

## 7. Remove the duplicated feature and refit

Remove `total_charges` and train the model again. If performance is unchanged, the feature was contributing nothing beyond confusion, and its removal is a decision that can be defended rather than a loss to be explained.

In [8]:
features_v2 = [f for f in X.columns if f != "total_charges"]

X_train2, X_test2 = X_train[features_v2], X_test[features_v2]

logreg2 = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
logreg2.fit(X_train2, y_train)

print("ROC AUC:", round(roc_auc_score(y_test, logreg2.predict_proba(X_test2)[:,1]), 3))

pd.DataFrame({
    "feature": X_train2.columns,
    "coefficient": logreg2.named_steps["model"].coef_[0]
}).sort_values("coefficient", ascending=False)


ROC AUC: 0.838


,feature,coefficient
1,monthly_charges,2.027357
4,contract_risk,0.727646
7,internet_service_No,0.328214
5,automatic_payment,-0.180307
6,internet_service_Fiber optic,-0.275306
2,has_phone,-0.588961
0,tenure_months,-0.742260
3,addon_count,-0.797284


**Result:** ROC AUC moved from 0.840 to 0.838 — effectively unchanged, confirming the feature contributed no predictive value.

More significantly, the `tenure_months` coefficient corrected from −1.22 to −0.74. That movement is the evidence: its value had been distorted by the duplicated feature, and it settled once the duplication was removed.

**The fibre coefficient remains negative, and this is not a fault.** A coefficient always describes an effect while holding all other features constant. A value of −0.28 therefore does not state that fibre customers churn less. It states that comparing a fibre customer and a DSL customer paying the same monthly amount, the fibre customer is marginally less likely to leave.

Fibre customers do churn more in absolute terms, but because they pay more, and `monthly_charges` has already accounted for that effect. The model has separated the influence of price from the influence of the product itself.

## 8. Verify the fibre finding directly

Query the raw churn rate by internet service type to confirm the interpretation above.

In [9]:
con.execute("""
    SELECT internet_service, COUNT(*) AS customers, ROUND(AVG(churn_value),3) AS churn_rate
    FROM vw_customer_features GROUP BY 1 ORDER BY 3 DESC
""").df()

,internet_service,customers,churn_rate
0,Fiber optic,3096,0.419
1,DSL,2421,0.190
2,No,1526,0.074


**Result:** the interpretation is confirmed.

| Internet service | Customers | Churn rate |
|---|---|---|
| Fibre optic | 3,096 | 41.9% |
| DSL | 2,421 | 19.0% |
| None | 1,526 | 7.4% |

Fibre customers churn at more than twice the DSL rate. The model attributes this to price rather than to the product, which carries a specific commercial implication: the issue lies in fibre pricing rather than in the fibre service.

> **Insight — fibre has a pricing problem, not a product problem.** Fibre churn of 41.9% would ordinarily prompt an investigation into service quality. Once price is held constant, fibre customers are marginally *less* likely to leave than DSL customers. The correct response is a pricing review, not a service review — a materially different and considerably cheaper intervention.

## 9. Build the gradient boosting model

Train the alternative model for comparison.

Rather than fitting a single equation, gradient boosting constructs a large number of small decision rules in sequence, each correcting the errors of those before it. This allows it to capture patterns a single equation cannot — for example, that high charges matter only for customers on month-to-month contracts.

No scaling is applied. Tree-based models divide the data at thresholds rather than measuring distances between values, so differences in units do not affect them.

In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X_train2, y_train)

y_pred_gb  = gb.predict(X_test2)
y_proba_gb = gb.predict_proba(X_test2)[:, 1]

print(classification_report(y_test, y_pred_gb, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba_gb), 3))

              precision    recall  f1-score   support

           0      0.828     0.894     0.860      1294
           1      0.624     0.486     0.546       467

    accuracy                          0.786      1761
   macro avg      0.726     0.690     0.703      1761
weighted avg      0.774     0.786     0.777      1761

ROC AUC: 0.829


**Result:** ROC AUC of 0.829, with recall of 48.6%.

**This comparison is not valid, and is retained deliberately to show why.** The logistic regression was trained with `class_weight="balanced"` and this model was not. It is therefore optimising for overall accuracy, which on imbalanced data favours predicting that customers will not churn. Its low recall is a consequence of that difference in settings, not of the method itself.

The only directly comparable figure at this stage is ROC AUC, which does not depend on where the decision threshold is placed.

## 10. Repeat the comparison under identical conditions

Retrain the gradient boosting model with the same class weighting applied to the logistic regression, so that the two are measured on equal terms.

In [11]:
gb = HistGradientBoostingClassifier(random_state=42, class_weight="balanced")
gb.fit(X_train2, y_train)

y_pred_gb  = gb.predict(X_test2)
y_proba_gb = gb.predict_proba(X_test2)[:, 1]

print(classification_report(y_test, y_pred_gb, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba_gb), 3))

              precision    recall  f1-score   support

           0      0.895     0.733     0.806      1294
           1      0.507     0.762     0.609       467

    accuracy                          0.740      1761
   macro avg      0.701     0.747     0.707      1761
weighted avg      0.792     0.740     0.754      1761

ROC AUC: 0.833


**Result:** with both models trained under identical conditions, the comparison is as follows.

| Measure | Logistic regression | Gradient boosting |
|---|---|---|
| ROC AUC | **0.838** | 0.833 |
| Recall — churners identified | **79.2%** | 76.2% |
| Precision | 50.5% | 50.7% |
| Coefficients available for inspection | Yes | No |

The simpler model performs marginally better on both headline measures, and it is the model whose decisions can be explained.

**A necessary qualification:** the two results are close enough that a different random division of the data could reverse the ordering. The defensible claim is not that logistic regression is superior, but that the additional complexity of gradient boosting delivers no measurable benefit on this dataset.

> **Insight — the added complexity bought nothing.** The usual trade-off is that an explainable model costs some accuracy. On this data there is no such cost to weigh: 0.838 against 0.833. The explainable model would have been the correct choice in a regulated setting even at a small disadvantage, and here it is not at one.

---

## 11. Score every customer

The evaluation above used only the 1,761 held-back customers. The economics layer requires a probability for all 7,043, so that value can be totalled across the whole customer base.

**A caveat that must be stated.** Most of these customers were used to train the model, so their probabilities are slightly optimistic — the model has already seen them. This is acceptable here because the purpose of this step is to demonstrate how the intervention decision is made, not to measure accuracy. Every performance figure quoted in this notebook comes from the held-back test set, which remains untouched.

Scoring the training data and presenting the result as clean performance is a common error, and a serious one. It is named here so that no figure produced downstream is mistaken for a measure of accuracy.

In [12]:
X_all = X[features_v2]

scored = df[["customer_id", "clv", "value_decile", "churn_value"]].copy()
scored["churn_probability"] = logreg2.predict_proba(X_all)[:, 1]

print(scored.shape)
scored.head()

(7043, 5)


,customer_id,clv,value_decile,churn_value,churn_probability
0,7569-NMZYQ,1282.50,1,0,0.127289
1,8984-HPEMB,1281.42,1,0,0.176850
2,5989-AXPUC,1280.88,1,0,0.189923
3,9924-JPRMC,1276.56,1,0,0.168200
4,3810-DVDQQ,1270.08,1,0,0.118914


**Result:** 7,043 customers scored, each with a churn probability alongside their lifetime value and value decile.

## 12. Write the scores to the database

Save the scored table so the economics notebook can read it without retraining anything.

This is stored as a table rather than a view, and it is the one place in the project where storing a result is the correct choice. Every other layer recalculates from source each time it runs. Model scores cannot: they are produced by a trained model that exists only while this notebook is running. The scores are therefore written down, and the notebook that produced them records how.

In [13]:
con.execute("DROP TABLE IF EXISTS customer_scores")
con.execute("CREATE TABLE customer_scores AS SELECT * FROM scored")

con.execute("SELECT COUNT(*) FROM customer_scores").df()

,count_star()
0,7043


## Summary

1. Nine features were carried into the model. The customer value fields were deliberately excluded so that churn risk and customer value remain separate questions, combined only at the decision stage.
2. A duplicated feature, `total_charges`, produced a coefficient contradicting an established finding. It was identified, removed, and confirmed to carry no predictive value. A second coefficient behaving unexpectedly was investigated and found to be correct once properly interpreted.
3. Both models were evaluated under identical conditions after the initial comparison was found to favour the preferred model unfairly.
4. Logistic regression is carried forward, achieving ROC AUC of 0.838 and identifying 79.2% of departing customers, with coefficients that can be inspected and explained.

**Next stage:** the intervention economics layer, combining the churn probabilities produced here with the customer lifetime values derived in SQL, to determine which customers are commercially worth retaining.